In [1]:
# 00 - Data Audit setup
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import BoundaryNorm, ListedColormap

def find_project_root(start=None):
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / 'config' / 'settings.py').exists():
            return candidate
    raise FileNotFoundError('Could not locate project root containing config/settings.py')

ROOT = find_project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from config.settings import (
    DATA_FINAL,
    DATA_PROCESSED,
    OUTPUTS_FIGURES,
    MIN_ARTICLES_FOR_SIGNAL,
    NUMPY_SEED,
    seed_everything,
)

seed_everything(NUMPY_SEED)

FIG_DIR = OUTPUTS_FIGURES / 'data_audit'
FIG_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option('display.max_rows', 200)
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 160)

plt.rcParams.update({
    'figure.dpi': 150,
    'savefig.dpi': 200,
    'savefig.bbox': 'tight',
    'font.size': 11,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'legend.fontsize': 10,
})

MASTER_PATH = DATA_FINAL / 'master_dataset.parquet'
RETURNS_PATH = DATA_PROCESSED / 'returns.parquet'

master_df = pd.read_parquet(MASTER_PATH)
returns_df = pd.read_parquet(RETURNS_PATH)

master_df['date'] = pd.to_datetime(master_df['date'])
returns_df['date'] = pd.to_datetime(returns_df['date'])

master_df = master_df.sort_values(['date', 'ticker']).reset_index(drop=True)
returns_df = returns_df.sort_values(['date', 'ticker']).reset_index(drop=True)

print(f'Loaded master_dataset: {len(master_df):,} rows, {len(master_df.columns)} columns')
print(f'Loaded returns dataset: {len(returns_df):,} rows, {len(returns_df.columns)} columns')

overlap = master_df[['date', 'ticker', 'ret_overnight', 'ret_intraday']].merge(
    returns_df[['date', 'ticker', 'ret_overnight', 'ret_intraday']],
    on=['date', 'ticker'],
    suffixes=('_master', '_returns'),
    how='inner',
)

overnight_diff = (overlap['ret_overnight_master'] - overlap['ret_overnight_returns']).abs().max()
intraday_diff = (overlap['ret_intraday_master'] - overlap['ret_intraday_returns']).abs().max()

print(f'Return overlap rows: {len(overlap):,}')
print(f'Max abs diff ret_overnight: {overnight_diff:.6f}')
print(f'Max abs diff ret_intraday: {intraday_diff:.6f}')

def normal_pdf(x, mean, std):
    x = np.asarray(x, dtype=float)
    if not np.isfinite(std) or std <= 0:
        return np.zeros_like(x)
    z = (x - mean) / std
    return np.exp(-0.5 * z ** 2) / (std * np.sqrt(2 * np.pi))


Loaded master_dataset: 61,740 rows, 18 columns
Loaded returns dataset: 78,398 rows, 13 columns
Return overlap rows: 61,740
Max abs diff ret_overnight: 0.000000
Max abs diff ret_intraday: 0.000000


In [2]:
# Section 1 - Coverage Heatmap

month_index = pd.period_range(
    master_df['date'].min().to_period('M'),
    master_df['date'].max().to_period('M'),
    freq='M',
).astype(str)

ticker_index = sorted(master_df['ticker'].dropna().unique())

coverage = (
    master_df.assign(month=master_df['date'].dt.to_period('M').astype(str))
    .groupby(['month', 'ticker'])['article_count_total']
    .sum()
    .unstack('ticker')
    .reindex(index=month_index, columns=ticker_index)
    .fillna(0)
    .astype(int)
)

cmap = ListedColormap(['#b91c1c', '#facc15', '#15803d'])
max_value = max(float(coverage.to_numpy().max()), 3.5)
norm = BoundaryNorm([-0.5, 0.5, 2.5, max_value + 0.5], cmap.N)

coverage_path = FIG_DIR / 'coverage_heatmap.png'
fig, ax = plt.subplots(figsize=(max(16, len(ticker_index) * 0.35), max(12, len(month_index) * 0.28)))
hm = sns.heatmap(
    coverage,
    ax=ax,
    cmap=cmap,
    norm=norm,
    linewidths=0.0,
    linecolor='white',
    cbar=True,
)
cbar = hm.collections[0].colorbar
cbar.set_ticks([0, 1, 2])
cbar.set_ticklabels(['0', '1-2', '3+'])
cbar.set_label('Article count band')

ax.set_title('Coverage Heatmap: article_count_total by stock-month')
ax.set_xlabel('Ticker')
ax.set_ylabel('YYYY-MM')
plt.setp(ax.get_xticklabels(), rotation=90, fontsize=7)
plt.setp(ax.get_yticklabels(), rotation=0, fontsize=7)
fig.tight_layout()
fig.savefig(coverage_path)
plt.close(fig)
print(f'Saved: {coverage_path}')

bad_month_share = (coverage <= 2).mean(axis=0)
bad_tickers = pd.DataFrame({
    'ticker': coverage.columns,
    'months_in_red_or_yellow': (coverage <= 2).sum(axis=0).values,
    'months_in_red': (coverage == 0).sum(axis=0).values,
    'months_in_yellow': ((coverage >= 1) & (coverage <= 2)).sum(axis=0).values,
    'share_red_or_yellow': bad_month_share.values,
    'share_red': (coverage == 0).mean(axis=0).values,
    'share_yellow': ((coverage >= 1) & (coverage <= 2)).mean(axis=0).values,
})
bad_tickers = bad_tickers[bad_tickers['share_red_or_yellow'] > 0.20].sort_values('share_red_or_yellow', ascending=False)

print('Tickers with >20% of months in red or yellow:')
if bad_tickers.empty:
    print('None')
else:
    display_tickers = bad_tickers.copy()
    for col in ['share_red_or_yellow', 'share_red', 'share_yellow']:
        display_tickers[col] = display_tickers[col].map(lambda x: f'{x:.1%}')
    print(display_tickers.to_string(index=False))

red_month_share = (coverage == 0).mean(axis=1)
bad_months = pd.DataFrame({
    'month': red_month_share.index,
    'red_ticker_count': (coverage == 0).sum(axis=1).values,
    'ticker_share_in_red': red_month_share.values,
})
bad_months = bad_months[bad_months['ticker_share_in_red'] > 0.30].sort_values('ticker_share_in_red', ascending=False)

print('Months with >30% of tickers in red:')
if bad_months.empty:
    print('None')
else:
    display_months = bad_months.copy()
    display_months['ticker_share_in_red'] = display_months['ticker_share_in_red'].map(lambda x: f'{x:.1%}')
    print(display_months.to_string(index=False))


Saved: C:\Users\ayush\OneDrive\Desktop\Theatre\Guido\Projects\Stock engine\outputs\figures\data_audit\coverage_heatmap.png
Tickers with >20% of months in red or yellow:
     ticker  months_in_red_or_yellow  months_in_red  months_in_yellow share_red_or_yellow share_red share_yellow
 SBILIFE.NS                       55             23                32               73.3%     30.7%        42.7%
HDFCLIFE.NS                       46             17                29               61.3%     22.7%        38.7%
 HCLTECH.NS                       22              4                18               29.3%      5.3%        24.0%
DIVISLAB.NS                       19              2                17               25.3%      2.7%        22.7%
Months with >30% of tickers in red:
None


In [3]:
# Section 2 - Return Distribution Check

return_cols = ['ret_overnight', 'ret_intraday']
hist_colors = {
    'ret_overnight': '#2563eb',
    'ret_intraday': '#0f766e',
}

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, col in zip(axes, return_cols):
    s = master_df[col].astype(float).dropna()
    mean = float(s.mean())
    std = float(s.std(ddof=1))
    skew = float(s.skew())
    kurt = float(s.kurt())
    extreme_count = int((s.abs() > 0.15).sum())

    print(f'{col}: mean={mean:.6f}, std={std:.6f}, skew={skew:.6f}, kurtosis={kurt:.6f}, |return|>15%={extreme_count:,}')

    ax.hist(s, bins=60, density=True, alpha=0.75, color=hist_colors[col], edgecolor='white', linewidth=0.4)
    x = np.linspace(float(s.min()), float(s.max()), 500)
    ax.plot(x, normal_pdf(x, mean, std), color='black', linewidth=2, label='Normal overlay')
    ax.set_title(col)
    ax.set_xlabel('Return')
    ax.set_ylabel('Density')
    ax.legend()

fig.tight_layout()
return_fig_path = FIG_DIR / 'return_distribution_check.png'
fig.savefig(return_fig_path)
plt.close(fig)
print(f'Saved: {return_fig_path}')

winsorized_df = master_df.copy()
rows_changed_mask = np.zeros(len(winsorized_df), dtype=bool)

for col in return_cols:
    lower = winsorized_df[col].quantile(0.01)
    upper = winsorized_df[col].quantile(0.99)
    clipped = winsorized_df[col].clip(lower=lower, upper=upper)
    changed = ~np.isclose(
        winsorized_df[col].to_numpy(dtype=float, copy=False),
        clipped.to_numpy(dtype=float, copy=False),
    )
    rows_changed_mask |= changed
    winsorized_df[col] = clipped
    print(f'{col}: 1st pct={lower:.6f}, 99th pct={upper:.6f}, clipped cells={int(changed.sum()):,}')

rows_winsorized = int(rows_changed_mask.sum())
print(f'Rows winsorized (any return column changed): {rows_winsorized:,}')

winsorized_df.to_parquet(MASTER_PATH, index=False)
master_df = winsorized_df
print(f'Overwrote: {MASTER_PATH}')


ret_overnight: mean=0.001469, std=0.009768, skew=-0.886065, kurtosis=21.212030, |return|>15%=0
ret_intraday: mean=-0.000578, std=0.018039, skew=0.180332, kurtosis=15.339129, |return|>15%=25
Saved: C:\Users\ayush\OneDrive\Desktop\Theatre\Guido\Projects\Stock engine\outputs\figures\data_audit\return_distribution_check.png
ret_overnight: 1st pct=-0.029951, 99th pct=0.028640, clipped cells=1,236
ret_intraday: 1st pct=-0.046225, 99th pct=0.050399, clipped cells=1,236
Rows winsorized (any return column changed): 2,203
Overwrote: C:\Users\ayush\OneDrive\Desktop\Theatre\Guido\Projects\Stock engine\data\final\master_dataset.parquet


In [5]:
# Section 3 - Signal Coverage

signal_cols = [
    'signal_organic_closed',
    'signal_organic_open',
    'signal_sponsored_closed',
    'signal_sponsored_open',
]

total_rows = len(master_df)
eligible_mask = master_df['article_count_total'] >= MIN_ARTICLES_FOR_SIGNAL
eligible_rows = int(eligible_mask.sum())

print(f'Total rows in master_dataset: {total_rows:,}')
print(f'Rows where article_count_total >= MIN_ARTICLES_FOR_SIGNAL ({MIN_ARTICLES_FOR_SIGNAL}): {eligible_rows:,}')

organic_closed_rows = int((eligible_mask & master_df['signal_organic_closed'].notna() & (master_df['signal_organic_closed'] != 0)).sum())
organic_open_rows = int((eligible_mask & master_df['signal_organic_open'].notna() & (master_df['signal_organic_open'] != 0)).sum())
sponsored_closed_rows = int((eligible_mask & master_df['signal_sponsored_closed'].notna() & (master_df['signal_sponsored_closed'] != 0)).sum())

print(f'Rows with signal_organic_closed not null/zero: {organic_closed_rows:,}')
print(f'Rows with signal_organic_open not null/zero: {organic_open_rows:,}')
print(f'Rows with signal_sponsored_closed not null/zero: {sponsored_closed_rows:,}')

all_four_ready_mask = eligible_mask & master_df[signal_cols].notna().all(axis=1)
all_four_count = int(all_four_ready_mask.sum())
print(f'Rows where ALL FOUR 2x2 cells are non-null: {all_four_count:,}')

if all_four_count < 500:
    print('WARNING: Insufficient observations for reliable 2x2 interaction test.')


Total rows in master_dataset: 61,740
Rows where article_count_total >= MIN_ARTICLES_FOR_SIGNAL (3): 49,627
Rows with signal_organic_closed not null/zero: 33,340
Rows with signal_organic_open not null/zero: 29,638
Rows with signal_sponsored_closed not null/zero: 44,964
Rows where ALL FOUR 2x2 cells are non-null: 23,041


In [6]:
# Section 4 - Temporal Coverage

monthly_articles = (
    master_df.assign(month=master_df['date'].dt.to_period('M').dt.to_timestamp())
    .groupby('month')['article_count_total']
    .sum()
    .sort_index()
)

monthly_fig_path = FIG_DIR / 'articles_per_month.png'
fig, ax = plt.subplots(figsize=(18, 6))
ax.axvspan(
    pd.Timestamp('2020-03-01'),
    pd.Timestamp('2020-07-01'),
    color='#fca5a5',
    alpha=0.25,
    label='COVID window',
)
ax.bar(
    monthly_articles.index,
    monthly_articles.values,
    width=20,
    color='#1d4ed8',
    edgecolor='white',
    linewidth=0.4,
)
ax.set_title('Total Articles per Month Across All Tickers')
ax.set_xlabel('Month')
ax.set_ylabel('Total articles')
ax.grid(axis='y', alpha=0.25)
ax.legend()
fig.autofmt_xdate()
fig.tight_layout()
fig.savefig(monthly_fig_path)
plt.close(fig)
print(f'Saved: {monthly_fig_path}')

top_5 = monthly_articles.sort_values(ascending=False).head(5)
bottom_5 = monthly_articles.sort_values(ascending=True).head(5)

print('5 highest article-count months:')
for month, value in top_5.items():
    print(f'  {month:%Y-%m}: {int(value):,}')

print('5 lowest article-count months:')
for month, value in bottom_5.items():
    print(f'  {month:%Y-%m}: {int(value):,}')


Saved: C:\Users\ayush\OneDrive\Desktop\Theatre\Guido\Projects\Stock engine\outputs\figures\data_audit\articles_per_month.png
5 highest article-count months:
  2024-04: 24,406
  2020-03: 23,142
  2020-07: 22,713
  2024-07: 22,690
  2023-05: 22,521
5 lowest article-count months:
  2025-06: 8,577
  2024-01: 8,910
  2020-11: 10,782
  2021-08: 12,252
  2021-11: 12,502


In [7]:
# Section 5 - Ready-to-Research Summary Table

total_stock_days = len(master_df)
days_ge_3_articles = eligible_rows
days_organic_closed = organic_closed_rows
days_organic_open = organic_open_rows
days_all_four_cells = all_four_count
usable_after_winsorization = int((~rows_changed_mask).sum())

summary_rows = [
    ('Total stock-days in master', total_stock_days, 100.0),
    ('Days with >= 3 articles', days_ge_3_articles, 100.0 * days_ge_3_articles / total_stock_days),
    ('Days with organic closed signal', days_organic_closed, 100.0 * days_organic_closed / total_stock_days),
    ('Days with organic open signal', days_organic_open, 100.0 * days_organic_open / total_stock_days),
    ('Days with ALL 4 cells (2x2 ready)', days_all_four_cells, 100.0 * days_all_four_cells / total_stock_days),
    ('Usable after winsorization', usable_after_winsorization, 100.0 * usable_after_winsorization / total_stock_days),
]

print('| Metric                              | Count  | % of Max |')
print('|-------------------------------------|--------|----------|')
for metric, count, pct in summary_rows:
    print(f'| {metric:<35} | {count:>6,} | {pct:>8.1f}% |')

if days_all_four_cells < 500:
    print('WARNING: Insufficient observations for reliable 2x2 interaction test.')


| Metric                              | Count  | % of Max |
|-------------------------------------|--------|----------|
| Total stock-days in master          | 61,740 |    100.0% |
| Days with >= 3 articles             | 49,627 |     80.4% |
| Days with organic closed signal     | 33,340 |     54.0% |
| Days with organic open signal       | 29,638 |     48.0% |
| Days with ALL 4 cells (2x2 ready)   | 23,041 |     37.3% |
| Usable after winsorization          | 59,537 |     96.4% |
